In [ ]:
from pathlib import Path

DATA_DIR  = Path('path_to_preprocessed_data')
TRAIN_DIR = DATA_DIR / 'train'
VALID_DIR = DATA_DIR / 'valid'
TEST_DIR  = DATA_DIR / 'test'

In [ ]:
import sys, os
REPO_PATH = 'path_to_repo'
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

from data.utils import get_datasets, precompute_features
from data.dataset import CachedDataset
from models.train import train, predict
from models.evaluate import evaluate
from models.transformer import Transformer
from models.utils import set_seed
from plots import plot_f1_training_curves, build_summary_df, plot_summary_table, plot_metrics_comparison, plot_confusion_matrix
from embeddings_plots import extract_embeddings, get_best_model, plot_tsne_comparison

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('Using device:', device)

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 30
BATCH_SIZE = 64
LR = 3e-4
REPR = 'mfcc'
D_MODELS = [64, 128, 256, 512]
CLASS_NAMES = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go','unknown', 'silence']

In [ ]:
train_ds, valid_ds, test_ds = get_datasets(data_format=REPR, train_path=TRAIN_DIR, valid_path=VALID_DIR, test_path=TEST_DIR)
precompute_features(train_ds, Path('/kaggle/working/cache/train_mfcc'))
precompute_features(valid_ds, Path('/kaggle/working/cache/valid_mfcc'))
precompute_features(test_ds,  Path('/kaggle/working/cache/test_mfcc'))

train_cached = CachedDataset('/kaggle/working/cache/train_mfcc')
valid_cached = CachedDataset('/kaggle/working/cache/valid_mfcc')
test_cached  = CachedDataset('/kaggle/working/cache/test_mfcc')

In [ ]:
def run_dmodel_experiment(d_models, seeds, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=device):
    """
    Trains and evaluates a Transformer model for each combination of d_model and seed.

    Returns:
        results[d_model][seed] = {
            'history', 'valid_acc', 'valid_loss', 'valid_f1',
            'test_acc', 'test_macro_f1', 'test_weighted_f1',
            'test_cm', 'model'
        }
    """
    results = {}

    for d_model in d_models:
        results[d_model] = {}
        for seed in seeds:
            print(f'\n[d_model={d_model}] seed={seed}')
            print('-' * 50)
            set_seed(seed)

            model = Transformer(n_features=40, n_timesteps=101, num_classes=12, d_model=d_model, nhead=4, num_layers=4,
                dropout=0.1, pooling='mean').to(device)

            model, history = train(model, train_cached, valid_cached, epochs=epochs, batch_size=batch_size, lr=lr, device=str(device),
                verbose=True, verbose_interval=5)

            preds, labels = predict(model, test_cached, device=str(device), batch_size=batch_size)
            test_results  = evaluate(preds, labels, print_report=False)

            results[d_model][seed] = {
                'history': history,
                'valid_acc': history['valid_acc'][-1],
                'valid_loss': history['valid_loss'][-1],
                'valid_f1': history['valid_f1'][-1],
                'test_acc': test_results['acc'],
                'test_macro_f1': test_results['macro_f1'],
                'test_weighted_f1': test_results['weighted_f1'],
                'test_cm': test_results['cm'],
                'model': model
            }

    return results

In [ ]:
dmodel_results = run_dmodel_experiment(D_MODELS, SEEDS)

In [ ]:
dmodel_results_str = {str(k): v for k, v in dmodel_results.items()}
plot_f1_training_curves(dmodel_results_str, title_prefix='Transformer d_model')

In [ ]:
summary_df = build_summary_df({
    f'd_model={k}': v for k, v in dmodel_results.items()
})
plot_summary_table(summary_df)

In [ ]:
plot_metrics_comparison(summary_df, title='Transformer — d_model Comparison')

In [ ]:
embeddings_per_dmodel = {}
for d in D_MODELS:
    print(f'Extracting embeddings for d_model={d}...')
    model = get_best_model(dmodel_results, d)
    embeddings, labels = extract_embeddings(model, test_cached, device)
    embeddings_per_dmodel[d] = {'embs': embeddings, 'labels': labels}
    print(f'  -> shape: {embeddings.shape}')

In [ ]:
plot_tsne_comparison(embeddings_per_dmodel, CLASS_NAMES)